# DL4M – FINAL Project (Group 8)
## Model Architecture & Training Pipeline
### Author: Zixuan Guo

---

This notebook presents the full training pipeline for our deep learning project:  **Noise-Robust Musical Instrument Classification**.

We use pre-trained YAMNet embeddings and a Temporal Convolutional Network (TCN) to classify solo instrument recordings under different SNR levels.  
The training process follows a **curriculum learning strategy**, starting from clean data and gradually fine-tuning on increasingly noisy data (SNR10 → SNR0 → SNR–5).

---

In [9]:
import utils as u
import models as m
import numpy as np
import os
import torch

**Set random seeds for reproducibility**

In [10]:
import warnings

# Fix the random seed for reproducibility
from numpy.random import seed
seed(124)

import tensorflow as tf
import keras
tf.keras.utils.set_random_seed(124)

### Load Yamnet and Map the data
**Load the YAMNet model from TensorFlow Hub**

In [11]:
import tensorflow_hub as hub

yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

**Extract embeddings and labels from the SNR5 dataset using YAMNet**

In [12]:
snr5_path = 'data/SNR5'

# Extract embeddings and labels from the SNR5 dataset using the YAMNet model
embeddings, labels = u.extract_embeddings_from_folder(snr5_path, yamnet)

print("Extraction completed.")
print("Embedding shape:", embeddings.shape)  # Expected shape: (num_frames, 1024)
print("Labels shape:", labels.shape)         # Expected shape: (num_frames,)

Processing cel...
Processing cla...
Processing flu...
Processing gac...
Processing gel...
Processing org...
Processing pia...
Processing sax...
Processing tru...
Processing vio...
Processing voi...
Extraction completed.
Embedding shape: (33822, 1024)
Labels shape: (33822,)


**Save the extracted embeddings and labels as .npy files**

In [13]:
np.save('SNR5_embeddings.npy', embeddings)
np.save('SNR5_labels.npy', labels)

print("Saved！")

Saved！


**Extract embeddings and labels from other dataset using YAMNet and save them**

In [15]:
# Process multiple noise levels and save their embeddings and labels
for snr_level in ['SNR10', 'SNR0', 'SNR-5']:
    print(f"Processing {snr_level}...")

    # Construct path to the current dataset folder (e.g., data/SNR10)
    folder_path = os.path.join('data', snr_level)

    # Extract embeddings and labels using the preloaded YAMNet model
    embeddings, labels = u.extract_embeddings_from_folder(folder_path, yamnet)

    # Directory to save the extracted features
    save_dir = 'embeddings'
    os.makedirs(save_dir, exist_ok=True)  # Create if it doesn't exist

    # Save embeddings and labels as .npy files
    np.save(os.path.join(save_dir, f'{snr_level}_embeddings.npy'), embeddings)
    np.save(os.path.join(save_dir, f'{snr_level}_labels.npy'), labels)

    print(f"{snr_level} saved successfully! ")

Processing SNR10...
Processing cel...
Processing cla...
Processing flu...
Processing gac...
Processing gel...
Processing org...
Processing pia...
Processing sax...
Processing tru...
Processing vio...
Processing voi...
SNR10 saved successfully! 
Processing SNR0...
Processing cel...
Processing cla...
Processing flu...
Processing gac...
Processing gel...
Processing org...
Processing pia...
Processing sax...
Processing tru...
Processing vio...
Processing voi...
SNR0 saved successfully! 
Processing SNR-5...
Processing cel...
Processing cla...
Processing flu...
Processing gac...
Processing gel...
Processing org...
Processing pia...
Processing sax...
Processing tru...
Processing vio...
Processing voi...
SNR-5 saved successfully! 


### Train the model using the clean dataset
**Load Preprocessed Embeddings and Labels**

In [ ]:
X = np.load('embeddings/SNR10_embeddings.npy')
y = np.load('embeddings/SNR10_labels.npy')

print(X.shape)  
print(y.shape)  

(33552, 1024)
(33552,)


**Build Sequence Data for TCN** 

In [ ]:
sequence_length = 60

X_seq, y_seq = u.create_sequences(X, y, sequence_length)

print(X_seq.shape)  
print(y_seq.shape)  

(33493, 60, 1024)
(33493,)


**Split Dataset into Train / Validation / Test: 80%, 10%, 10%**

In [ ]:
from sklearn.model_selection import train_test_split


X_train, X_val_test, y_train, y_val_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42, shuffle=True
)


X_val, X_test, y_val, y_test = train_test_split(
    X_val_test, y_val_test, test_size=0.5, random_state=42, shuffle=True
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (26794, 60, 1024) (26794,)
Val: (3349, 60, 1024) (3349,)
Test: (3350, 60, 1024) (3350,)


**Wrap Data into TensorFlow Datasets**

In [ ]:
batch_size = 64

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_dataset = test_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

2025-04-26 13:46:15.483964: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2025-04-26 13:46:15.483993: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-04-26 13:46:15.483997: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2025-04-26 13:46:15.484013: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-26 13:46:15.484023: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


**Define and Train the TCN Model and Save Trained Model Checkpoint**

In [ ]:
model = m.tcn_model(input_shape=(60, 1024))
model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset
)
model.save('checkpoints/tcn_snr10_model.h5')

/opt/anaconda3/lib/python3.12/site-packages/tcn/tcn.py:227: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


Epoch 1/10


2025-04-26 13:46:22.194603: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


419/419 ━━━━━━━━━━━━━━━━━━━━ 37s 78ms/step - accuracy: 0.8145 - loss: 0.5752 - val_accuracy: 0.9952 - val_loss: 0.0238
Epoch 2/10
419/419 ━━━━━━━━━━━━━━━━━━━━ 32s 77ms/step - accuracy: 0.9938 - loss: 0.0228 - val_accuracy: 0.9976 - val_loss: 0.0105
Epoch 3/10
419/419 ━━━━━━━━━━━━━━━━━━━━ 30s 72ms/step - accuracy: 0.9959 - loss: 0.0171 - val_accuracy: 0.9934 - val_loss: 0.0256
Epoch 4/10
419/419 ━━━━━━━━━━━━━━━━━━━━ 30s 72ms/step - accuracy: 0.9970 - loss: 0.0101 - val_accuracy: 0.9970 - val_loss: 0.0097
Epoch 5/10
419/419 ━━━━━━━━━━━━━━━━━━━━ 30s 73ms/step - accuracy: 0.9974 - loss: 0.0124 - val_accuracy: 0.9970 - val_loss: 0.0106
Epoch 6/10
419/419 ━━━━━━━━━━━━━━━━━━━━ 31s 74ms/step - accuracy: 0.9992 - loss: 0.0028 - val_accuracy: 0.9893 - val_loss: 0.0314
Epoch 7/10
419/419 ━━━━━━━━━━━━━━━━━━━━ 31s 73ms/step - accuracy: 0.9926 - loss: 0.0297 - val_accuracy: 0.9979 - val_loss: 0.0100
Epoch 8/10
419/419 ━━━━━━━━━━━━━━━━━━━━ 31s 73ms/step - accuracy: 0.9975 - loss: 0.0086 - val_accurac

**Evaluate Model on Test Set**

In [ ]:
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.9985 - loss: 0.0036
Test Loss: 0.0021
Test Accuracy: 0.9994


### Curriculum Learning: Fine-tune on SNR5

Following the baseline training on clean SNR10 data, we now apply curriculum learning by fine-tuning on noisier SNR5 data.

The workflow mirrors the previous training steps:
- Load precomputed SNR5 embeddings and labels
- Segment the data into overlapping sequences
- Split into train/val/test sets
- Prepare TensorFlow Datasets

We then load the previously trained TCN model (`tcn_snr10_model.h5`), freeze the backbone layers, and fine-tune only the final dense layer to adapt the model to a noisier environment.

In [ ]:
X_snr5 = np.load('embeddings/SNR5_embeddings.npy')
y_snr5 = np.load('embeddings/SNR5_labels.npy')

print(X_snr5.shape, y_snr5.shape)

(33822, 1024) (33822,)


In [ ]:
X_snr5_seq, y_snr5_seq = u.create_sequences(X_snr5, y_snr5, sequence_length=60)

print(X_snr5_seq.shape, y_snr5_seq.shape)

(33763, 60, 1024) (33763,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val_test, y_train, y_val_test = train_test_split(X_snr5_seq, y_snr5_seq, test_size=0.2, random_state=42, shuffle=True)
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, test_size=0.5, random_state=42, shuffle=True)

batch_size = 64

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

2025-04-26 13:55:36.389450: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2025-04-26 13:55:36.389482: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-04-26 13:55:36.389487: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2025-04-26 13:55:36.389509: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-26 13:55:36.389522: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [ ]:
# Load the previously trained model on SNR10 data
# Freeze all layers except the final dense output layer
# Recompile the model before fine-tuning on SNR5

model = model = m.load_model('checkpoints/tcn_snr10_model.h5', custom_objects={'TCN': m.TCN})
for layer in model.layers[:-1]:  
    layer.trainable = False


model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset
)
model.save('checkpoints/tcn_snr5_model.h5')

Epoch 1/10


2025-04-26 13:55:57.785604: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


423/423 ━━━━━━━━━━━━━━━━━━━━ 18s 36ms/step - accuracy: 0.9571 - loss: 0.2994 - val_accuracy: 0.9541 - val_loss: 0.1583
Epoch 2/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 13s 31ms/step - accuracy: 0.9550 - loss: 0.1468 - val_accuracy: 0.9588 - val_loss: 0.1305
Epoch 3/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 13s 31ms/step - accuracy: 0.9611 - loss: 0.1219 - val_accuracy: 0.9573 - val_loss: 0.1224
Epoch 4/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 13s 31ms/step - accuracy: 0.9611 - loss: 0.1172 - val_accuracy: 0.9597 - val_loss: 0.1146
Epoch 5/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 13s 31ms/step - accuracy: 0.9636 - loss: 0.1098 - val_accuracy: 0.9609 - val_loss: 0.1105
Epoch 6/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 13s 32ms/step - accuracy: 0.9655 - loss: 0.1030 - val_accuracy: 0.9588 - val_loss: 0.1127
Epoch 7/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 13s 32ms/step - accuracy: 0.9657 - loss: 0.0979 - val_accuracy: 0.9648 - val_loss: 0.1072
Epoch 8/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 14s 33ms/step - accuracy: 0.9659 - loss: 0.0987 - val_accurac

In [ ]:
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.9668 - loss: 0.1035
Test Loss: 0.1007
Test Accuracy: 0.9659


### Curriculum Learning: Fine-tune on SNR0

As the next step in curriculum learning, we move from clean data (SNR10) and moderate noise (SNR5) to a more challenging environment — the SNR0 dataset.  
We load the precomputed embeddings and labels, then convert them into 60-frame sequences for training.

In [ ]:
X_snr0 = np.load('embeddings/SNR0_embeddings.npy')
y_snr0 = np.load('embeddings/SNR0_labels.npy')

print(X_snr0.shape, y_snr0.shape)

(33870, 1024) (33870,)


In [ ]:
X_snr0_seq, y_snr0_seq = u.create_sequences(X_snr0, y_snr0, sequence_length=60)

print(X_snr0_seq.shape, y_snr0_seq.shape)

(33811, 60, 1024) (33811,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val_test, y_train, y_val_test = train_test_split(X_snr0_seq, y_snr0_seq, test_size=0.2, random_state=42, shuffle=True)
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, test_size=0.5, random_state=42, shuffle=True)

batch_size = 64

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

2025-04-26 14:02:13.026381: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2025-04-26 14:02:13.026407: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-04-26 14:02:13.026411: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2025-04-26 14:02:13.026429: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-26 14:02:13.026440: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [ ]:
# Load the previously trained model on SNR5 data
# Freeze all layers except the final dense output layer
# Compile the model before fine-tuning on SNR0

model = m.load_model('checkpoints/tcn_snr5_model.h5', custom_objects={'TCN': m.TCN})
for layer in model.layers[:-1]: 
    layer.trainable = False

# compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset
)
model.save('checkpoints/tcn_snr0_model.h5')

Epoch 1/10


2025-04-26 14:02:19.411700: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


423/423 ━━━━━━━━━━━━━━━━━━━━ 37s 77ms/step - accuracy: 0.9539 - loss: 0.1613 - val_accuracy: 0.9956 - val_loss: 0.0166
Epoch 2/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 30s 72ms/step - accuracy: 0.9937 - loss: 0.0203 - val_accuracy: 0.9947 - val_loss: 0.0172
Epoch 3/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 32s 77ms/step - accuracy: 0.9984 - loss: 0.0068 - val_accuracy: 0.9979 - val_loss: 0.0099
Epoch 4/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 31s 74ms/step - accuracy: 0.9992 - loss: 0.0037 - val_accuracy: 0.9698 - val_loss: 0.0951
Epoch 5/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 33s 77ms/step - accuracy: 0.9955 - loss: 0.0158 - val_accuracy: 0.9976 - val_loss: 0.0074
Epoch 6/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 34s 80ms/step - accuracy: 0.9972 - loss: 0.0109 - val_accuracy: 0.9962 - val_loss: 0.0160
Epoch 7/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 35s 82ms/step - accuracy: 0.9984 - loss: 0.0059 - val_accuracy: 0.9994 - val_loss: 0.0035
Epoch 8/10
423/423 ━━━━━━━━━━━━━━━━━━━━ 33s 79ms/step - accuracy: 0.9994 - loss: 0.0029 - val_accurac

In [ ]:
# Evaluate
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.9999 - loss: 4.0373e-04
Test Loss: 0.0014
Test Accuracy: 0.9994


### Curriculum Learning: Fine-tune on SNR–5

In this final stage of curriculum learning, we load the SNR–5 dataset, which contains the noisiest audio samples.  
As before, we convert the embeddings into overlapping sequences with aligned labels to prepare for model fine-tuning.  
This stage challenges the model's robustness in extremely low signal-to-noise conditions.

In [ ]:
X_snr_minus5 = np.load('embeddings/SNR-5_embeddings.npy')
y_snr_minus5 = np.load('embeddings/SNR-5_labels.npy')

print(X_snr_minus5.shape, y_snr_minus5.shape)

(33936, 1024) (33936,)


In [ ]:
X_snr_minus5_seq, y_snr_minus5_seq = u.create_sequences(X_snr_minus5, y_snr_minus5, sequence_length=60)

print(X_snr_minus5_seq.shape, y_snr_minus5_seq.shape)

(33877, 60, 1024) (33877,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val_test, y_train, y_val_test = train_test_split(X_snr_minus5_seq, y_snr_minus5_seq, test_size=0.2, random_state=42, shuffle=True)
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, test_size=0.5, random_state=42, shuffle=True)

batch_size = 64

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

2025-04-26 14:09:49.001991: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2025-04-26 14:09:49.002023: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-04-26 14:09:49.002027: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2025-04-26 14:09:49.002047: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-26 14:09:49.002059: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [ ]:
# Load the model previously fine-tuned on SNR0
# Freeze all layers except the final dense output layer
# Recompile the model before continuing fine-tuning on noisier SNR-5 data

model = m.load_model('checkpoints/tcn_snr0_model.h5', custom_objects={'TCN': m.TCN})
for layer in model.layers[:-1]: 
    layer.trainable = False


model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset
)
model.save('checkpoints/tcn_snr-5_model.h5')

Epoch 1/10


2025-04-26 14:09:56.492274: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


424/424 ━━━━━━━━━━━━━━━━━━━━ 38s 79ms/step - accuracy: 0.9319 - loss: 0.2758 - val_accuracy: 0.9950 - val_loss: 0.0215
Epoch 2/10
424/424 ━━━━━━━━━━━━━━━━━━━━ 32s 75ms/step - accuracy: 0.9957 - loss: 0.0192 - val_accuracy: 0.9956 - val_loss: 0.0141
Epoch 3/10
424/424 ━━━━━━━━━━━━━━━━━━━━ 31s 72ms/step - accuracy: 0.9943 - loss: 0.0208 - val_accuracy: 0.9953 - val_loss: 0.0137
Epoch 4/10
424/424 ━━━━━━━━━━━━━━━━━━━━ 32s 75ms/step - accuracy: 0.9982 - loss: 0.0060 - val_accuracy: 0.9994 - val_loss: 0.0011
Epoch 5/10
424/424 ━━━━━━━━━━━━━━━━━━━━ 31s 73ms/step - accuracy: 0.9989 - loss: 0.0049 - val_accuracy: 0.9985 - val_loss: 0.0039
Epoch 6/10
424/424 ━━━━━━━━━━━━━━━━━━━━ 30s 72ms/step - accuracy: 0.9979 - loss: 0.0077 - val_accuracy: 0.9994 - val_loss: 0.0038
Epoch 7/10
424/424 ━━━━━━━━━━━━━━━━━━━━ 30s 70ms/step - accuracy: 0.9992 - loss: 0.0023 - val_accuracy: 0.9970 - val_loss: 0.0158
Epoch 8/10
424/424 ━━━━━━━━━━━━━━━━━━━━ 30s 70ms/step - accuracy: 0.9973 - loss: 0.0102 - val_accurac

In [ ]:
# Evaluate
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 1.0000 - loss: 4.6204e-04
Test Loss: 0.0007
Test Accuracy: 1.0000
